In [41]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier

In [ ]:
data = pd.read_csv("Telco-Customer-Churn.csv")
print(data.head())

data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data.drop("customerID", axis=1, inplace=True)

print(data.isnull().sum())

data = data.dropna()

print(data.info())
print(data.shape)


## Data Preprocessing

Bu aşamada Telco Customer Churn veri seti sisteme yüklenmiştir. TotalCharges değişkeni normalde sayısal bir değişken olmasına rağmen veri setinde object (string) tipinde bulunduğu için nümerik formata dönüştürülmüştür. Daha sonra eksik veri kontrolü yapılmış ve yalnızca TotalCharges değişkeninde 11 adet eksik değer olduğu görülmüştür. Bu sayı toplam veri setine göre oldukça düşük olduğu için eksik gözlemler silinmiştir.
Ayrıca customerID değişkeni yalnızca müşteri kimliği içerdiği ve model tahminine katkı sağlamadığı için veri setinden çıkarılmıştır.

In [ ]:
print(data["Churn"].value_counts())
print(data["Churn"].value_counts(normalize=True) * 100)

sns.countplot(x="Churn", data=data)
plt.title("Churn Distribution")
plt.show()

features = ["tenure", "MonthlyCharges", "TotalCharges"]
plt.figure(figsize=(12,8))
for i, col in enumerate(features, 1):
    plt.subplot(2,2,i)
    sns.boxplot(x="Churn", y=col, data=data)
    plt.title(f"{col} vs Churn")
plt.tight_layout()
plt.show()

sns.countplot(x="Contract", hue="Churn", data=data)
plt.title("Contract Type vs Churn")
plt.xticks(rotation=15)
plt.show()

sns.countplot(x="PaymentMethod", hue="Churn", data=data)
plt.title("Payment Method vs Churn")
plt.xticks(rotation=45)
plt.show()

sns.countplot(x="OnlineSecurity", hue="Churn", data=data)
plt.title("Online Security vs Churn")
plt.show()

numeric_data = data.select_dtypes(include=["int64", "float64"])
plt.figure(figsize=(10,6))
sns.heatmap(numeric_data.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

sns.boxplot(x=data["tenure"])
plt.title("Tenure Outlier Value Review")
plt.show()

sns.boxplot(x=data["MonthlyCharges"])
plt.title("Monthly Charges Outlier Value Review")
plt.show()

sns.boxplot(x=data["TotalCharges"])
plt.title("Total Charges Outlier Value Review")
plt.show()

## EDA (Exploratory Data Analysis)

Churn dağılım grafiğinden veri setinin dengesiz olduğu ve churn etmeyen müşteri sayısının fazla olduğu görülmüştür. Sayısal değişkenler açısından tenure, MonthlyCharges ve TotalCharges değişkenleri churn ile karşılaştırılmıştır. Özellikle düşük tenure değerine sahip müşterilerin churn oranının daha yüksek olduğu gözlemlenmiştir. Ayrıca aylık ödeme tutarı yüksek olan müşterilerde churn riski artış göstermektedir. 

In [44]:
data['AverageMonthlyCharge'] = data['TotalCharges'] / (data['tenure'] + 1) 

data["IsNewCustomer"] = np.where(data["tenure"] < 12, 1, 0)

data["LongTermCustomer"] = np.where(data["tenure"] > 24, 1, 0)

data["HighRiskContract"] = np.where(data["Contract"] == "Month-to-month", 1, 0)

data["NoSupportRisk"] = np.where((data["TechSupport"] == "No") & (data["OnlineSecurity"] == "No"), 1, 0)

## Feature Engineering

Bu aşamada modelin müşteri kaybını daha iyi tahmin edebilmesi için mevcut değişkenlerden yeni ve anlamlı özellikler üretilmiştir. 

AverageMonthlyCharge: Müşterinin toplam harcamasının müşteri süresine bölünmesiyle oluşturulmuştur. Bu değişken, müşterinin ortalama aylık harcama davranışını daha net analiz etmeyi amaçlamaktadır. 

IsNewCustomer: 12 aydan kısa süredir hizmet alan müşteriler yeni müşteri olarak tanımlanmıştır. Yeni müşterilerin churn etme olasılığı genellikle daha yüksek olduğu için bu değişken modele önemli katkı sağlamaktadır.

LongTermCustomer: 24 aydan uzun süredir sistemde bulunan müşteriler uzun dönem müşteri olarak tanımlanmıştır. Uzun dönem müşterilerin churn etme olasılığı daha azdır.

HighRiskContract: month-to-month kontrata sahip müşteriler belirlenmiştir. EDA analizinde bu müşteri grubunun churn riskinin daha yüksek olduğu gözlemlendiği için bu bilgi modele eklenmiştir. 

NoSupportRisk: Hem teknik destek almayan hem de online güvenlik hizmeti kullanmayan müşteriler belirlenmiştir. Bu müşterilerin memnuniyet seviyesinin daha düşük olabileceği ve churn riskinin artabileceği düşünülmüştür.

In [45]:
data['gender'] = data['gender'].map({'Female': 1, 'Male': 0})
data['Partner'] = data['Partner'].map({'Yes': 1, 'No': 0})
data['Dependents'] = data['Dependents'].map({'Yes': 1, 'No': 0})
data['PhoneService'] = data['PhoneService'].map({'Yes': 1, 'No': 0})
data['PaperlessBilling'] = data['PaperlessBilling'].map({'Yes': 1, 'No': 0})
data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0}) 

data = pd.get_dummies(data, drop_first=True)

data.to_csv("processed_telco_data.csv", index=False)

## Encoding 

İki sınıflı (binary) değişkenler olan gender, Partner, Dependents, PhoneService, PaperlessBilling ve hedef değişken olan Churn, Yes/No ve Female/Male yapısına göre 0 ve 1 değerlerine dönüştürülmüştür. Birden fazla kategoriye sahip olan değişkenler için ise pd.get_dummies() yöntemi kullanılmıştır. Bu yöntem sayesinde kategorik değişkenler modelin anlayabileceği ayrı sütunlara dönüştürülmüştür.

Encoding işlemi tamamlandıktan sonra veri seti modelleme aşamasında kullanılmak üzere `processed_telco_data.csv` adıyla kaydedilmiştir.


In [46]:
data = pd.read_csv("processed_telco_data.csv")

## Load Process Data

Kaydedilen temiz veri yüklenmiştir.

In [ ]:
X = data.drop("Churn", axis=1)
y = data["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
y_train.value_counts().plot(kind='bar')
plt.title("Train set class distribution")
plt.xlabel("Churn")

plt.subplot(1,2,2)
y_test.value_counts().plot(kind='bar')
plt.title("Test set class distribution")
plt.tight_layout()
plt.show()


## Train/Test Split

Veri seti bağımsız değişkenler (X) ve hedef değişken (y) olarak ayrılmıştır. Hedef değişken olarak müşteri kaybını ifade eden Churn kullanılmıştır.
Model performansını daha iyi değerlendirmek için veri seti train ve test seti olarak bölünmüştür. stratify=y kullanılarak churn sınıf dağılımının hem train hem de test setinde eşit olması sağlanmıştır.

In [ ]:
print(f"before SMOTE: {y_train.value_counts()}")

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"After SMOTE: {y_train_res.value_counts()}")

## SMOTE

EDA aşamasında churn eden müşterilerin veri setinde azınlık sınıfını oluşturduğu görülmüştür. Churn etmeyen müşteriler çoğunlukta olduğu için sınıf dengesizliği problemi oluşmaktadır. Bu problemi çözmek amacıyla SMOTE yöntemi kullanlmıştır. SMOTE, azınlık sınıfına ait sentetik örnekler üreterek veri setindeki sınıf dağılımını dengelemektedir. SMOTE uygulanmadan önce churn eden müşteri sayısı daha düşükken, işlem sonrasında her iki sınıf eşit hale getirilmiştir. Bu sayede modelin churn müşterilerini daha iyi öğrenmesi ve Recall performansının artırılması hedeflenmiştir.

In [49]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

## Scaling

Logistic Regression modeli ölçek duyarlı (scale-sensitive) bir algoritma olduğu için bu model eğitilirken StandardScaler kullanılmıştır. Bu işlem sayesinde tüm sayısal değişkenler benzer ölçeğe getirilmiş ve modelin daha sağlıklı öğrenmesi sağlanmıştır. Random Forest ve LightGBM modelleri tree_based olduğu için scaling kullanılmamıştır.

In [50]:
#Logistic Regression
lr_model = LogisticRegression(max_iter=1000,random_state=42)
lr_model.fit(X_train_scaled, y_train_res)
lr_pred = lr_model.predict(X_test_scaled)

#Random Forest
rf_model = RandomForestClassifier(n_estimators=200,random_state=42)
rf_model.fit(X_train_res, y_train_res)
rf_pred = rf_model.predict(X_test)

#LightGBM 
lgbm_model = LGBMClassifier(n_estimators=200,random_state=42,learning_rate=0.05,verbose=-1)
lgbm_model.fit(X_train_res, y_train_res) 
lgbm_pred = lgbm_model.predict(X_test)

## Model Trainnig

Üç farklı sınıflandırma modeli eğitilmiştir: Logistic Regression, Random Forest ve LightGBM. Amaç farklı algoritmaların churn tahmin performanslarını karşılaştırmak ve en uygun modeli belirlemektir.

In [ ]:
models = {
    "Logistic Regression": lr_pred,
    "Random Forest": rf_pred,
    "LightGBM": lgbm_pred
}

performance_metrics = []

for name, pred in models.items():
    performance_metrics.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1-Score": f1_score(y_test, pred),
        "AUC-ROC": roc_auc_score(y_test, pred)
    })

perf_df = pd.DataFrame(performance_metrics)
print("\n--- Model Comparisİon Table ---")
print(perf_df)

## Model Comparision Table

Burada Accuracy, Precision, Recall, F1-score ve AUC-ROC metrikleri kullanılarak modeller karşılaştırılmıştır. Churn prediction problemlerinde yalnızca accuracy değerine bakmak yanıltıcı olabilir. Çünkü asıl amaç, sistemi terk edecek müşterileri mümkün olduğunca doğru tespit edebilmektir. Bu nedenle Recall metriği daha kritik kabul edilmiştir. Sonuçlara göre Logistic Regression modeli en yüksek recall değerine sahip olmasına rağmen precision değerinin daha düşük olduğu görülmüştür. Random Forest modeli ise recall ve precision arasında daha dengeli bir performans göstermiştir. Özellikle F1-Score değerinin en yüksek olması, modelin genel sınıflandırma başarısının daha güçlü olduğunu göstermektedir.

Bu nedenle proje kapsamında en başarılı model olarak Random Forest seçilmiş ve sonraki analizler bu model üzerinden gerçekleştirilmiştir.

In [ ]:
cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.title("Random Forest - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Confusion Matrix

Model comparision table soncunda en iyi model çıkan Random Forest modelinin Confusion Matrix ile doğru ve yanlış tahminleri detaylı olarak incelenmiştir. Özellikle false negative değerinin düşük olması önemlidir çünkü churn edecek müşteriyi kaçırmak şirket için yüksek maliyet yaratır.
False positive durumunda ise aslında churn etmeyecek bir müşteriye kampanya veya teklif sunulmuş olur. u durum ek maliyet yaratsa da false negative kadar kritik değildir. Bu nedenle model değerlendirilirken yalnızca accuracy değil, özellikle recall ve false negative oranı dikkate alınmıştır. Random Forest modelinin bu açıdan daha dengeli ve güvenilir sonuç verdiği görülmüştür.

In [ ]:
risk_scores = rf_model.predict_proba(X_test)[:, 1]

risk_df = X_test.copy()
risk_df["Churn_Probability"] = risk_scores

top_risk_customers = risk_df.sort_values(
    by="Churn_Probability",
    ascending=False
).head(int(len(risk_df) * 0.10))

print("\n===== Top 10% Risk Customers =====")
print(top_risk_customers.head(10))

## Risk Customers Analysis

Random Forest modeli kullanılarak test veri setindeki müşterilerin churn olasılıkları hesaplanmıştır. predict_proba() fonksiyonu sayesinde her müşteri için churn etme ihtimali olasılık bazlı olarak elde edilmiştir. Bu skorlar kullanılarak churn riski en yüksek olan ilk %10’luk müşteri grubu belirlenmiştir. Bu segment, şirket açısından öncelikli olarak aksiyon alınması gereken kritik müşteri grubunu temsil etmektedir. Bu müşterilere özel kampanyalar, indirimler, sadakat programları veya kişiselleştirilmiş teklifler sunularak müşteri kaybının azaltılması hedeflenmektedir.

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
plt.figure(figsize=(10,6))
importances.nlargest(10).plot(kind='barh', color='teal')
plt.title("The 10 Most Influential Characteristics in the Decision-Making Process")
plt.show()

## Feature Importance 

Random Forest modeli kullanılarak churn tahmininde en etkili olan değişkenler analiz edilmiştir. Feature Importance yöntemi sayesinde modelin karar verme sürecinde hangi değişkenlerin daha belirleyici olduğu incelenmiştir. Sonuçlara göre sözleşme tipi (Contract), müşteri süresi (tenure), aylık ödeme tutarı (MonthlyCharges) ve destek hizmetleri gibi değişkenlerin churn üzerinde önemli etkiler yarattığı görülmüştür. Özellikle month-to-month kontrata sahip müşteriler, kısa süreli müşteriler ve teknik destek almayan kullanıcılar daha yüksek churn riski taşımaktadır.

## Yönetici Notu

Model sonuçlarına göre müşteri kaybını en çok etkileyen faktörler sözleşme tipi (Contract), müşteri süresi (tenure), aylık ödeme tutarı (MonthlyCharges), teknik destek (TechSupport) ve online güvenlik hizmetleri (OnlineSecurity) olarak belirlenmiştir.

### 1. Kontrat Seçenekleri
   Özellikle month-to-month kontrata sahip müşterilerin churn etme olasılığının daha yüksek olduğu görülmüştür. Bu müşteri grubu uzun vadeli bağlılık göstermediği için rakip firmalara geçiş yapma eğilimi daha fazladır. Bu nedenle 1 yıllık ve 2 yıllık kontrat seçenekleri daha avantajlı fiyatlarla sunulabilir. Uzun dönem kontratlarda indirim uygulanması, ücretsiz ek hizmetler verilmesi müşteri elde tutma oranını artırabilir.

### 2. Teknik Destek ve Güvenlik Teşvikleri
   Model, TechSupport ve OnlineSecurity gibi ek hizmetleri kullanmayan müşterilerin daha kolay ayrıldığı bulunmuştur. Bu hizmetleri hiç kullanmamış müşterilere 3 aylık ücretsiz deneme paketleri sunulmalıdır. Ayrıca özel destek paketleri veya güvenlik hizmetlerinde kampanya sunulması müşteri memnuniyetini artırabilir.

### 3. Yeni Müşteri
   Yeni müşterilerin özellikle ilk 12 ay içerisinde churn etme olasılığı yüksektir. Bu nedenle ilk 6-12 ay içerisinde müşteri memnuniyeti anketleri gönderilmeli ve gelen sikayetlere geri dönüş sağlanmalıdır ve kişileştirilmiş kampanyalarla müşteri memnuniyeti sağlanmalıdır. 

### 4. Ödeme Yöntemi
   Elektronik çek ile ödeme yapanlarda churn oranı, otomatik ödeme yapan müşterilere göre daha yüksektir. Müşteriler, kredi kartı veya banka hesabı üzerinden otomatik ödeme talimatı vermeye teşvik edilmelidir. Otomatik ödemeye geçen müşterilere küçük çaplı tek seferlik indirimler veya puanlar verilmesi, ödeme kaynaklı churn oranlarını azaltabilir.